# Chapter 3: PyTorch Lightning — Removing Training-Loop Boilerplate

### Why This Matters

Chapter 2's explicit training loop is instructive to see once, but writing it out fully, correctly (with checkpointing, multi-GPU support, logging, and early stopping all handled by hand) for every single project is genuinely tedious and error-prone. **PyTorch Lightning** keeps your model definition as ordinary PyTorch, but organizes the surrounding *engineering* — the training loop itself, device management, logging, checkpointing — into a standard, reusable structure.

In [ ]:
import lightning as L

class LitModel(L.LightningModule):
    def __init__(self, n_features):
        super().__init__()
        self.model = SimpleNet(n_features)     # reuse the exact PyTorch nn.Module from Chapter 2
        self.criterion = nn.BCELoss()

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred = self.model(x)
        loss = self.criterion(pred, y)
        self.log("train_loss", loss)             # automatic logging — no manual print/plot bookkeeping
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

trainer = L.Trainer(max_epochs=30, accelerator="auto")     # "auto" -> uses a GPU/TPU automatically if available
trainer.fit(LitModel(n_features), train_loader)

Notice what disappeared compared to Chapter 2's explicit loop: no manual `for epoch in range(...)`, no manual `zero_grad()/backward()/step()` sequence, no manual device placement (`accelerator="auto"` handles moving data and the model to a GPU automatically if one's available). You define **what** happens in one training step (`training_step`) and **how** to optimize (`configure_optimizers`); Lightning's `Trainer` handles **how** the surrounding loop, logging, and hardware are actually orchestrated. This separation of "research logic" from "engineering machinery" is exactly why Lightning has become a standard choice for serious, reproducible deep learning projects — the same `LitModel` class trains identically whether you're on a laptop CPU or a multi-GPU cluster, with zero code changes.

### Cheat Sheet — PyTorch Lightning

| Task | Code |
|---|---|
| Define a model | `class LitModel(L.LightningModule): training_step / configure_optimizers` |
| Train | `L.Trainer(max_epochs=N, accelerator="auto").fit(model, train_loader)` |
| Log a metric | `self.log("name", value)` |
| Automatic checkpointing | `L.Trainer(callbacks=[ModelCheckpoint()])` |

---